In [1]:
from dotenv import load_dotenv
from llm_zoomcamp_2026_code.ingest import load_faq_data, build_index
import json
from openai import OpenAI


load_dotenv()


openai_client = OpenAI()

## Indexing the documents


In [2]:

documents = load_faq_data()
index = build_index(documents)

## Model without Any Context and Tools

In [3]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Absolutely — if the course is still open, you should be able to join it.\n\nIf you want, I can help you figure out:\n- whether registration is still available,\n- how to enroll,\n- or what to say if you need to ask the instructor directly.\n\nIf you send me the course name or a link, I can help more specifically.'

## Defining the Tool

In [9]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

## Running LLM with the Tool

In [6]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered the course join now enrollment late start"}', call_id='call_oyp4Dl0Cs3mPmMaHk7ZDZvw7', name='search', type='function_call', id='fc_0eca3d01a1032055006aca5a09df3487d19213aea27ea323a3', async_=None, caller=None, namespace=None, status='completed')]

In [ ]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2) #Got the context required for LLM to answer the query. 

{'query': 'Can I join the course late discovered the course join now enrollment late start'}

## keep storing everything in the message History

In [ ]:
messages.extend(response.output)

# storing the function call output in a specific format
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [12]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered the course join now enrollment late start"}', call_id='call_oyp4Dl0Cs3mPmMaHk7ZDZvw7', name='search', type='function_call', id='fc_0eca3d01a1032055006aca5a09df3487d19213aea27ea323a3', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_oyp4Dl0Cs3mPmMaHk7ZDZvw7',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and 

## Finally Asking the model with the Context

In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join. The course materials are available, and you can start anytime.\n\nIf you want a certificate, though, you’ll need to submit your project while submissions are still open.'

## Token Usage

In [14]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(787, 44)

In [15]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(usage.input_tokens, usage.output_tokens)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.00014445


In [ ]:


def encode(strs):

    if strs==[]:
        return ''
        
    else:
        return ('').join(strs)+'_' + str(('#').join([str(len(i)) for i in strs]))



def decode(s):

    if s=='':
        return []
    else:

        x= s.split('_')
        a,b = x[0],x[1]
        result = []
        curr =0 
        b  = b.split('#')
        for i in range(len(b)):
        
            next = curr+int(b[i])
            result.append(a[curr:next])
            curr = next
            

        return result


In [34]:
strs=["","   ","!@#$%^&*()_+","LongStringWithNoSpaces","Another, String With, Commas"]
a = encode(strs)
print(a)
#decode(a)

   !@#$%^&*()_+LongStringWithNoSpacesAnother, String With, Commas_0#3#12#22#28


In [20]:
s = 'HelloWorld_55'

In [ ]:
x= s.split('_')
a,b = x[0],x[1]



0 5 Hello
5 10 World


In [38]:
decode('HelloWorld_55')

['', '']

In [18]:
a='HelloWorld_55'
b = a.split('_')[1]

In [20]:
for i in b:
    print(i)

5
5


In [5]:
strs = 'HelloWorld_55_aadadda'

print(strs.rsplit('_',1))

['HelloWorld_55', 'aadadda']


In [6]:
strs.lsplit('_',1)

AttributeError: 'str' object has no attribute 'lsplit'

In [9]:
# A string with multiple hyphens
text = "user-profile-image-v2.jpg"

# Standard split vs rsplit with maxsplit=1
print(text.split("-"))   # Output: ['user', 'profile-image-v2.jpg']
print(text.rsplit("-"))  # Output: ['user-profile-image', 'v2.jpg']


['user', 'profile', 'image', 'v2.jpg']
['user', 'profile', 'image', 'v2.jpg']


In [21]:
nums = [-1,0,1,2,3]

st =1
end =1
size = len(nums)-1
pre_list = []
post_list = []
final_result = []
for i in range(len(nums)):

    pre_list.append(st)
    st=st*nums[i]
    post_list.append(end)

    end = end * nums[size]
    size = size-1

size = len(pre_list)
print(size)
for i in range(size):

    final_result.append(pre_list[i]*post_list[size-1])
    size = size-1

print(final_result)




5
[0, -6, 0, 0, 0]


In [26]:
board =[["1","2",".",".","3",".",".",".","."],
 ["4",".",".","5",".",".",".",".","."],
 [".","9","8",".",".",".",".",".","3"],
 ["5",".",".",".","6",".",".",".","4"],
 [".",".",".","8",".","3",".",".","5"],
 ["7",".",".",".","2",".",".",".","6"],
 [".",".",".",".",".",".","2",".","."],
 [".",".",".","4","1","9",".",".","8"],
 [".",".",".",".","8",".",".","7","9"]]

In [27]:
# for i in board:
#     print(i)

In [22]:
def hasDuplicate(nums):
        hash_map = dict()
        for i in nums:
            if i !='.':
                if i not in hash_map:
                    hash_map[i]=1
                else:
                    return True
        return False

In [99]:
def solve_sudoku(board):

    hash_map = dict()
    columns =[]
    for i in range(9):
        columns.append(['.'])
        for j in range(9):
            x= i//3
            y = j//3
            elem = board[i][j]
            columns[i].extend(board[j][i])
            if elem!='.':
                if (x,y) not in hash_map:
                    hash_map[(x,y)]=[board[i][j]]
                else:
                    if elem not in hash_map[(x,y)]:
                        hash_map[(x,y)].extend(board[i][j])
                    else:
                        #print(hash_map)
                        return False
            else:
                pass
    
    for i in board:
        if hasDuplicate(i):
            return False

    for j in columns:
        if hasDuplicate(j):
            return False
        
    return True 

In [100]:
board=[[".",".","4",".",".",".","6","3","."],[".",".",".",".",".",".",".",".","."],["5",".",".",".",".",".",".","9","."],[".",".",".","5","6",".",".",".","."],["4",".","3",".",".",".",".",".","1"],[".",".",".","7",".",".",".",".","."],[".",".",".","5",".",".",".",".","."],[".",".",".",".",".",".",".",".","."],[".",".",".",".",".",".",".",".","."]]

In [101]:
for i in board:
    print(i)

['.', '.', '4', '.', '.', '.', '6', '3', '.']
['.', '.', '.', '.', '.', '.', '.', '.', '.']
['5', '.', '.', '.', '.', '.', '.', '9', '.']
['.', '.', '.', '5', '6', '.', '.', '.', '.']
['4', '.', '3', '.', '.', '.', '.', '.', '1']
['.', '.', '.', '7', '.', '.', '.', '.', '.']
['.', '.', '.', '5', '.', '.', '.', '.', '.']
['.', '.', '.', '.', '.', '.', '.', '.', '.']
['.', '.', '.', '.', '.', '.', '.', '.', '.']


In [102]:
print(len(board),len(board[0]))

9 9


In [103]:
solve_sudoku(board)

False

In [97]:
len(board)

9

In [98]:
len(board[0])

9

In [13]:
hash_map[(0,0)]

['1', '2', '.', '4', '.', '.', '.', '9', '8']

In [12]:
s = "Was it a car or a cat I saw"
new_str= ''
for i in s:
    if i.isalnum():
        new_str = new_str+i.lower()


print(new_str)
start = 0
end = len(new_str)-1

while (start<end):

    if new_str[start] == new_str[end]:
        start=start+1
        end = end-1
    else:
        print(new_str[start],new_str[end],False)
        break

print(True)


wasitacaroracatisaw
True


In [16]:


nums=[9,1,4,7,3,-1,0,5,8,-1,6]

nums = list(set(nums))
print(nums)

result = []
hash_map = dict()

c=1
# for i in range(len(nums)):

#     if nums[i] not in hash_map:
#         if nums[i]-c in nums:
#             if nums[i]-c not in hash_map:
#                 hash_map[nums[i]-c]=[nums[i]]
#                 print(hash_map)
#             else:
#                 hash_map[nums[i]-c].append(nums[i])
#             c=c-1
#         else:
#             result.append(hash_map)
#             hash_map = dict()
            

for i in range(len(nums)):
    if nums[i] not in hash_map:
        if nums[i]-1 not in nums:
            hash_map[nums[i]]=[]

for i in nums:
    if i-c in hash_map:
        hash_map[i-c].append(i)
    c=c+1

[0, 1, 3, 4, 5, 6, 7, 8, 9, -1]


In [17]:
hash_map

{3: [], -1: [0, 1]}

In [30]:

nums=[0,2,4,6,8,10,12,14,16,18,20,22,24,26,28,30,32,34,36,38,40,42,44,46,48,50,52,54,56,58,60,62,64,66,68,70,72,74,76,78,80,82,84,86,88,90,92,94,96,98,100,102,104,106,108,110,112,114,116,118,120,122,124,126,128,130,132,134,136,138,140,142,144,146,148,150,152,154,156,158,160,162,164,166,168,170,172,174,176,178,180,182,184,186,188,190,192,194,196,198,200,202,204,206,208,210,212,214,216,218,220,222,224,226,228,230,232,234,236,238,240,242,244,246,248,250,252,254,256,258,260,262,264,266,268,270,272,274,276,278,280,282,284,286,288,290,292,294,296,298,300,302,304,306,308,310,312,314,316,318,320,322,324,326,328,330,332,334,336,338,340,342,344,346,348,350,352,354,356,358,360,362,364,366,368,370,372,374,376,378,380,382,384,386,388,390,392,394,396,398,400,402,404,406,408,410,412,414,416,418,420,422,424,426,428,430,432,434,436,438,440,442,444,446,448,450,452,454,456,458,460,462,464,466,468,470,472,474,476,478,480,482,484,486,488,490,492,494,496,498,500,502,504,506,508,510,512,514,516,518,520,522,524,526,528,530,532,534,536,538,540,542,544,546,548,550,552,554,556,558,560,562,564,566,568,570,572,574,576,578,580,582,584,586,588,590,592,594,596,598,600,602,604,606,608,610,612,614,616,618,620,622,624,626,628,630,632,634,636,638,640,642,644,646,648,650,652,654,656,658,660,662,664,666,668,670,672,674,676,678,680,682,684,686,688,690,692,694,696,698,700,702,704,706,708,710,712,714,716,718,720,722,724,726,728,730,732,734,736,738,740,742,744,746,748,750,752,754,756,758,760,762,764,766,768,770,772,774,776,778,780,782,784,786,788,790,792,794,796,798,800,802,804,806,808,810,812,814,816,818,820,822,824,826,828,830,832,834,836,838,840,842,844,846,848,850,852,854,856,858,860,862,864,866,868,870,872,874,876,878,880,882,884,886,888,890,892,894,896,898,900,902,904,906,908,910,912,914,916,918,920,922,924,926,928,930,932,934,936,938,940,942,944,946,948,950,952,954,956,958,960,962,964,966,968,970,972,974,976,978,980,982,984,986,988,990,992,994,996,998,1000,1002,1004,1006,1008,1010,1012,1014,1016,1018,1020,1022,1024,1026,1028,1030,1032,1034,1036,1038,1040,1042,1044,1046,1048,1050,1052,1054,1056,1058,1060,1062,1064,1066,1068,1070,1072,1074,1076,1078,1080,1082,1084,1086,1088,1090,1092,1094,1096,1098,1100,1102,1104,1106,1108,1110,1112,1114,1116,1118,1120,1122,1124,1126,1128,1130,1132,1134,1136,1138,1140,1142,1144,1146,1148,1150,1152,1154,1156,1158,1160,1162,1164,1166,1168,1170,1172,1174,1176,1178,1180,1182,1184,1186,1188,1190,1192,1194,1196,1198,1200,1202,1204,1206,1208,1210,1212,1214,1216,1218,1220,1222,1224,1226,1228,1230,1232,1234,1236,1238,1240,1242,1244,1246,1248,1250,1252,1254,1256,1258,1260,1262,1264,1266,1268,1270,1272,1274,1276,1278,1280,1282,1284,1286,1288,1290,1292,1294,1296,1298,1300,1302,1304,1306,1308,1310,1312,1314,1316,1318,1320,1322,1324,1326,1328,1330,1332,1334,1336,1338,1340,1342,1344,1346,1348,1350,1352,1354,1356,1358,1360,1362,1364,1366,1368,1370,1372,1374,1376,1378,1380,1382,1384,1386,1388,1390,1392,1394,1396,1398,1400,1402,1404,1406,1408,1410,1412,1414,1416,1418,1420,1422,1424,1426,1428,1430,1432,1434,1436,1438,1440,1442,1444,1446,1448,1450,1452,1454,1456,1458,1460,1462,1464,1466,1468,1470,1472,1474,1476,1478,1480,1482,1484,1486,1488,1490,1492,1494,1496,1498,1500,1502,1504,1506,1508,1510,1512,1514,1516,1518,1520,1522,1524,1526,1528,1530,1532,1534,1536,1538,1540,1542,1544,1546,1548,1550,1552,1554,1556,1558,1560,1562,1564,1566,1568,1570,1572,1574,1576,1578,1580,1582,1584,1586,1588,1590,1592,1594,1596,1598,1600,1602,1604,1606,1608,1610,1612,1614,1616,1618,1620,1622,1624,1626,1628,1630,1632,1634,1636,1638,1640,1642,1644,1646,1648,1650,1652,1654,1656,1658,1660,1662,1664,1666,1668,1670,1672,1674,1676,1678,1680,1682,1684,1686,1688,1690,1692,1694,1696,1698,1700,1702,1704,1706,1708,1710,1712,1714,1716,1718,1720,1722,1724,1726,1728,1730,1732,1734,1736,1738,1740,1742,1744,1746,1748,1750,1752,1754,1756,1758,1760,1762,1764,1766,1768,1770,1772,1774,1776,1778,1780,1782,1784,1786,1788,1790,1792,1794,1796,1798,1800,1802,1804,1806,1808,1810,1812,1814,1816,1818,1820,1822,1824,1826,1828,1830,1832,1834,1836,1838,1840,1842,1844,1846,1848,1850,1852,1854,1856,1858,1860,1862,1864,1866,1868,1870,1872,1874,1876,1878,1880,1882,1884,1886,1888,1890,1892,1894,1896,1898,1900,1902,1904,1906,1908,1910,1912,1914,1916,1918,1920,1922,1924,1926,1928,1930,1932,1934,1936,1938,1940,1942,1944,1946,1948,1950,1952,1954,1956,1958,1960,1962,1964,1966,1968,1970,1972,1974,1976,1978,1980,1982,1984,1986,1988,1990,1992,1994,1996,1998]

In [ ]:

highest = 1

if len(nums)>1:
    nums_set = set(nums)
    hash_map = dict() 

    for i in nums: 
        if i-1 not in nums_set:
            hash_map[i]=1

    for k in hash_map.keys():
        key_temp = k+1
        while key_temp in nums_set:
            hash_map[k]=hash_map[k]+1
            if hash_map[k]>highest:
                highest=hash_map[k]
            key_temp=key_temp+1

else:
    print(highest)



In [32]:
print(hash_map)
print(highest)

{0: 1, 2: 1, 4: 1, 6: 1, 8: 1, 10: 1, 12: 1, 14: 1, 16: 1, 18: 1, 20: 1, 22: 1, 24: 1, 26: 1, 28: 1, 30: 1, 32: 1, 34: 1, 36: 1, 38: 1, 40: 1, 42: 1, 44: 1, 46: 1, 48: 1, 50: 1, 52: 1, 54: 1, 56: 1, 58: 1, 60: 1, 62: 1, 64: 1, 66: 1, 68: 1, 70: 1, 72: 1, 74: 1, 76: 1, 78: 1, 80: 1, 82: 1, 84: 1, 86: 1, 88: 1, 90: 1, 92: 1, 94: 1, 96: 1, 98: 1, 100: 1, 102: 1, 104: 1, 106: 1, 108: 1, 110: 1, 112: 1, 114: 1, 116: 1, 118: 1, 120: 1, 122: 1, 124: 1, 126: 1, 128: 1, 130: 1, 132: 1, 134: 1, 136: 1, 138: 1, 140: 1, 142: 1, 144: 1, 146: 1, 148: 1, 150: 1, 152: 1, 154: 1, 156: 1, 158: 1, 160: 1, 162: 1, 164: 1, 166: 1, 168: 1, 170: 1, 172: 1, 174: 1, 176: 1, 178: 1, 180: 1, 182: 1, 184: 1, 186: 1, 188: 1, 190: 1, 192: 1, 194: 1, 196: 1, 198: 1, 200: 1, 202: 1, 204: 1, 206: 1, 208: 1, 210: 1, 212: 1, 214: 1, 216: 1, 218: 1, 220: 1, 222: 1, 224: 1, 226: 1, 228: 1, 230: 1, 232: 1, 234: 1, 236: 1, 238: 1, 240: 1, 242: 1, 244: 1, 246: 1, 248: 1, 250: 1, 252: 1, 254: 1, 256: 1, 258: 1, 260: 1, 262: 1